# **Semantic Web & Linked Open Data with Python**
## บทที่ 04 — Logics

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rathachai/semantic-web/blob/main/code/ch04/ch04_notebook.ipynb)

> ## *Author* : [Rathachai CHAWUTHAI](https://rathachai.creatier.pro/) , Ph.D
> ### *Affiliation* : Computer Engineering, King Mongkut's Institute of Technology Ladkrabang (KMITL)
> #### *Updated Date* : 2026-08-23
---

<a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Creative Commons License" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/88x31.png" /></a><br />This work (course text) is licensed under a <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Creative Commons Attribution 4.0 International License</a>. Sample code cells are additionally released under the MIT License.

**References:**
*   https://owlready2.readthedocs.io/
*   http://www.hermit-reasoner.com/

📖 เอกสารประกอบบทนี้: [`04-logics/`](https://github.com/Rathachai/semantic-web/tree/main/04-logics)

---

### Setup

`owlready2` ต้องมี **Java** เพื่อรัน HermiT reasoner

In [1]:
!pip install -q owlready2

In [2]:
import shutil, subprocess

if shutil.which('java') is None:
    print('ไม่พบ Java -- กำลังติดตั้ง (ใช้เวลาราว 30 วินาที บน Colab)...')
    subprocess.run(['apt-get', '-qq', 'install', '-y',
                    'openjdk-17-jre-headless'], check=False)

print('java:', shutil.which('java') or 'ไม่พบ -- reasoning cell ด้านล่างอาจรันไม่ได้')

java: /usr/bin/java


In [3]:
from owlready2 import *
print('owlready2 พร้อมใช้งาน')

owlready2 พร้อมใช้งาน


## To Create a TBox (นิยาม class)

แปลง DL axiom `Student ⊑ Person` และ `Parent ≡ Person ⊓ ∃hasChild.Person` เป็นโค้ด Python ตรง ๆ

In [4]:
onto = get_ontology('http://example.org/uni#')

In [5]:
with onto:
    class Person(Thing): pass
    class Student(Person): pass
    class Course(Thing): pass

    class hasChild(ObjectProperty):
        domain = [Person]; range = [Person]

    class enrolledIn(ObjectProperty):
        domain = [Student]; range = [Course]

    class Parent(Person):
        equivalent_to = [Person & hasChild.some(Person)]

print("TBox สร้างเสร็จ")

TBox สร้างเสร็จ


## To Create an ABox (ข้อเท็จจริง)

In [6]:
with onto:
    alice = Student("alice")
    bob = Person("bob")
    alice.hasChild = [bob]
    cs501 = Course("cs501")
    alice.enrolledIn = [cs501]

print(alice.is_a)

[uni.Student]


## Query ก่อนรัน reasoner

`alice` ยังไม่ถูกจัดว่าเป็น `Parent` เพราะเป็นแค่ equivalentClass ที่ยังไม่ได้อนุมาน

In [7]:
alice.is_a

[uni.Student]

## รัน HermiT Reasoner

reasoner จะคำนวณ **subsumption**, **instance checking**, และ **realization** ให้

In [8]:
sync_reasoner_hermit([onto], infer_property_values=True)

* Owlready2 * Running HermiT...
    java -Xmx2000M -cp /Users/rathachai/opt/anaconda3/lib/python3.9/site-packages/owlready2/hermit:/Users/rathachai/opt/anaconda3/lib/python3.9/site-packages/owlready2/hermit/HermiT.jar org.semanticweb.HermiT.cli.CommandLine -c -O -D -I file:////var/folders/p7/wrfb851s61sg68ywkl0vr3n00000gn/T/tmp759lipde -Y
* Owlready2 * HermiT took 1.108921766281128 seconds
* Owlready * Reparenting uni.alice: {uni.Student} => {uni.Parent, uni.Student}
* Owlready * (NB: only changes on entities loaded in Python are shown, other changes are done but not listed)


In [9]:
alice.is_a   # ตอนนี้ควรมี Parent ปรากฏด้วย

[uni.Student, uni.Parent]

## Reasoning Task ที่ 1: Instance Checking

"alice เป็น Parent หรือไม่?"

In [10]:
Parent in alice.is_a

True

## Reasoning Task ที่ 2: Realization

"ใครบ้างที่เป็น Parent?"

In [11]:
list(Parent.instances())

[uni.alice]

## Reasoning Task ที่ 3: KB Consistency

ลองสร้างความขัดแย้งโดยตั้งใจ — ประกาศว่า `Student` กับ `Course` เป็น **disjoint** แล้วให้ individual ตัวหนึ่งเป็นทั้งคู่

In [12]:
with onto:
    AllDisjoint([Student, Course])
    alice.is_a.append(Course)   # ⚠️ ทำให้ขัดแย้งโดยตั้งใจ

try:
    sync_reasoner_hermit([onto])
    print("✅ ยังคง consistent (ไม่ควรเกิดขึ้น)")
except OwlReadyInconsistentOntologyError:
    print("❌ ตรวจพบ inconsistency ตามคาด -- alice เป็นทั้ง Student และ Course พร้อมกันไม่ได้")

* Owlready2 * Running HermiT...
    java -Xmx2000M -cp /Users/rathachai/opt/anaconda3/lib/python3.9/site-packages/owlready2/hermit:/Users/rathachai/opt/anaconda3/lib/python3.9/site-packages/owlready2/hermit/HermiT.jar org.semanticweb.HermiT.cli.CommandLine -c -O -D -I file:////var/folders/p7/wrfb851s61sg68ywkl0vr3n00000gn/T/tmpbert6ifo


❌ ตรวจพบ inconsistency ตามคาด -- alice เป็นทั้ง Student และ Course พร้อมกันไม่ได้


## 🎯 สรุป

- **TBox** = นิยาม class/property, **ABox** = ข้อเท็จจริงของ individual
- `equivalent_to` แปลงจาก DL `≡` ตรงตัว
- reasoner ทำ **instance checking**, **realization**, **consistency checking** ให้อัตโนมัติ
- `AllDisjoint` + ข้อมูลที่ขัดกัน → **`OwlReadyInconsistentOntologyError`**

ลองเพิ่ม individual คนใหม่ที่มี `hasChild` แล้วรัน reasoner อีกครั้ง — จะกลายเป็น Parent ทันที!

---

## จบบทที่ 04 🎉

ไปต่อบทถัดไป: **บทที่ 05 — Knowledge Management**

📚 กลับหน้าคอร์ส: [https://github.com/Rathachai/semantic-web](https://github.com/Rathachai/semantic-web)

---
https://github.com/Rathachai/semantic-web

*Semantic Web & Linked Open Data with Python*